# Checking the simulation from outside it

> *"How do you know the simulation is right, and not merely self-consistent?"*

Notebooks 01–06 all run the same way: a trajectory becomes a parametric waveform, the waveform
becomes pupil terms through the Eqs. S5–S6 Taylor expansion, and the terms become closed-form
Gaussian fields. Fast, exact for quadratic phase — and entirely self-consistent, which is
precisely the problem. **A sign error shared by the synthesizer and the simulator is invisible
to every test that goes through both.**

This page runs the other path. `aodl.check` takes the **rendered RF samples** — the literal AWG
buffers, carrier included — measures the drive back off them with an FFT, rebuilds the aperture
field with no expansion at all, and propagates it to the image plane with a chirp-z transform.
It shares with the simulator exactly two things: `params.py` and the sign table in
`device/conventions.py`.

| § | What you get |
|---|--------------|
| 1 | why the loop has to be closed from outside, and how |
| 2 | the flagship drive, checked — `plan.check()` → PASS |
| 3 | inside the checker: the record's spectrum, the aperture's order comb, the band window |
| 4 | the crystal's own compression, $(2J_1(C)/C)^2$, measured from samples |
| 5 | measured vs requested trajectory, with the $t-\tau/2$ alignment and the tolerance band |
| 6 | the blob audit during a hand-over: what is allowed to be lit, and what is not |
| 7 | breaking it on purpose — a flipped `Ax` chirp, and the report that catches it |

Physics reference: arXiv:2510.11451 (`S#` = its Supplement). The figures in §3 are the first
FFTs anywhere in this repository: the simulation path has none, by design (`CLAUDE.md`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import j1

from aodl import ArraySpec, Lift, TrajectorySpec, Translate, default_1030, plan_motion
from aodl.check import ApertureGrid, Tolerances, band_window, demodulate, from_arrays
from aodl.check.demod import out_of_band_fraction, sample_baseband
from aodl.check.report import frame_reach
from aodl.device import conventions
from aodl.units import MHz, mm, um, us
from aodl.waveform.export import DEFAULT_SAMPLE_RATE, render_samples

P = default_1030()
tau = P.channels["Ax"].transit_time

# ---- the flagship of docs/guide.md 2: 10x10, lift 10 um, traverse 40 x 25 um, drop ----
story = TrajectorySpec(
    array=ArraySpec(10, 10, delta_f_x=1.0 * MHz, delta_f_y=1.3 * MHz),
    moves=(Lift(10 * um, 150 * us), Translate(40 * um, 25 * um, 250 * us), Lift(-10 * um, 150 * us)),
)
plan = plan_motion(story, P)
print(f"mode {plan.report.mode}, {plan.report.n_tones} tones, "
      f"{len(plan.report.fade_events)} hand-overs, T = {story.duration / us:.0f} us")
print(f"check frames [us]: {np.round(plan.check_times() / us, 2)}")

## 1. What the checker actually does

```
   rendered samples  V_mu[k]          (the AWG buffer: carrier included, globally normalized)
        |
        |  one FFT per channel: z = 2 P+[V] e^{-i 2 pi f_c t}      demod.py   (Eqs. S1-S2)
        v
   complex baseband  z_mu(t)
        |
        |  gather z at the *retarded* time of every aperture point,
        |  T = exp(i C V), FFT along u, keep the +1 band, IFFT       pupil.py  (Eqs. S1-S4)
        v
   aperture field  P(u)                                    du = Lambda/8, +-4.99 w_in
        |
        |  chirp-z transform onto the image grid, + Eq. S11 defocus  transform.py
        v
   U(X), U(Y) per sub-time  ->  <|U_x|^2 (x) |U_y|^2>  ->  fits      metrics.py / report.py
```

Three choices are worth naming, because they are what make the answer trustworthy rather than
merely different:

* **the aperture is sampled at $\Lambda/8$**, eight cells per acoustic wavelength. $e^{iCV}$ is
  a phase-modulated carrier whose spectrum is a comb of diffraction orders; sampling at $8/\Lambda$
  makes every alias land on an order *centre*, so the first one to fold onto $+1$ is $|J_7|$ —
  $2\times10^{-9}$ at the product drive. §3 shows the comb.
* **the order is selected in the aperture's spatial-frequency domain**, which is what an AODL
  does optically: the orders are separated *angles*. No expansion order is truncated, so the
  compression and every intermodulation product come along for free (§4).
* **nothing is evaluated at one instant.** Atoms and cameras see the intensity averaged over the
  MHz beat notes between pupil terms, so each frame averages over a window — measured in the
  array's own moving frame, so that a fast traverse is not reported as a smear.

## 2. The flagship, checked

One call. It renders the drive to float64 samples at 625 MS/s, builds the expectation from the
trajectory and Table I alone, and rebuilds a hundred tweezers at seven frames spanning the
550 µs manoeuvre.

The waist and uniformity gates are opened for this particular drive, and the reason is the
drive: a 30-rung ladder at `drive_strength = 0.30` renders with a **normalization factor**
(peak over single-tone amplitude) of 4.59, so the crystal's peak modulation index is 1.4 rad
and Eqs. S20–S22 spread the per-trap intensity by about a fifth. That spread is not purely
$C^2$: about 82 % of it scales that way and the rest is a ~3.8 % $C$-independent floor from the
fade-speed apodization (`docs/guide.md` §5.5). §4 measures the $C^2$ part at its root. The
*position* gates run at their defaults.

In [ ]:
report = plan.check(k_subtimes=48, tolerances=Tolerances(waist=0.12, uniformity=0.30))
print(report.summary())
assert report.passed

Read the residual block from the top. **Lateral 0.01 $w_0$** — a hundred tweezers, tracked
through a 3D move, sit within a hundredth of a spot of where they were asked to be.
**Astigmatism 0.02 $z_R$**, against a Table I prediction of exactly zero: that is the paper's
central claim, measured from the RF buffer rather than from the model that wrote it.
**No off-lattice light at all** — every blob the audit found sits on the extended Shepard
lattice, which is where a fading ladder is supposed to put it (§6).

Two lines below the residuals say what the verdict is *about*. **`coverage 64 %`**: a fading
array's two edge lines on each axis leave the intensity gates (§6.6), so the 10×10's outer ring
is exempt and 8×8 traps are judged — a fading 2×2 would have nothing left at all, and
`Tolerances(require_coverage=True)` makes that a failure rather than a quiet pass.
**`uniformity_median`** is the same deviation medianed over the seven frames: 0.137, and
report-only, because on a Shepard drive the ladder slides and a rung fault visits a different
column at every frame instead of holding still (`docs/guide.md` §5.5).

## 3. Inside the checker: two spectra

The left panel is the **record**: the literal `Bx` buffer, Hann-windowed and transformed. The
ten live rungs of the Shepard ladder sit inside the ±10 MHz band around the 100 MHz carrier,
and the skirts outside it are the Table II rectangles switching on and off — `p_B = 0` rungs
have no ramp, so they radiate (`switch_ramp` removes it; notebook 05 §8).

The right panel is the **aperture**: the spatial-frequency spectrum of $e^{iCV}$ on the pinned
$\Lambda/8$ grid. Orders sit $f_c/v$ apart, four of them inside Nyquist, and the shaded window
is what `channel_pupil` keeps. The cells below re-do by hand exactly what `pupil.py` does,
which is the point of showing them.

In [ ]:
frame = float(plan.check_times()[3])                       # mid-traverse
grid = ApertureGrid.design(P, "bragg_band")
reach = frame_reach(grid, P)
span = (max(0.0, frame - 0.5 * tau - reach - 5 * us), frame - 0.5 * tau + reach + 5 * us)
arrays, scale = render_samples(plan.wfs, DEFAULT_SAMPLE_RATE, span, dtype=np.float64,
                               return_scale=True)
rec = from_arrays(arrays, DEFAULT_SAMPLE_RATE, P, t_start=span[0], normalization=scale)
bb = demodulate(rec)
print(f"{rec.n_samples} samples per channel over "
      f"[{span[0] / us:.1f}, {span[1] / us:.1f}] us; normalization {rec.normalization:.3f}")
print("out-of-band power fraction: "
      + ", ".join(f"{k} {v:.2e}" for k, v in out_of_band_fraction(rec).items()))

# the aperture, exactly as pupil.py builds it (Eqs. S1-S4)
aod, geom = P.channels["Bx"], conventions.geometry("Bx")
t_ret = conventions.retarded_time(frame, grid.u, geom, aod)
drive = np.real(sample_baseband(bb, "Bx", t_ret) * np.exp(2j * np.pi * aod.f_center * t_ret))
transmission = np.where(conventions.is_filled(grid.u, frame, geom, aod),
                        np.exp(1j * aod.drive_strength * drive), 1.0)
apodized = transmission * np.exp(-((grid.u / P.optics.w_in) ** 2))
nu = np.fft.fftshift(np.fft.fftfreq(grid.n, grid.du))
orders = np.abs(np.fft.fftshift(np.fft.fft(apodized)))
centre = geom.sound_sign * aod.f_center / P.sound_speed
half = 1.15 * 0.5 * (aod.band[1] - aod.band[0]) / P.sound_speed

In [ ]:
fig, (ax_r, ax_a) = plt.subplots(1, 2, figsize=(11.6, 3.9))

taper = np.hanning(rec.n_samples)
freq = np.fft.rfftfreq(rec.n_samples, 1.0 / rec.sample_rate)
power = np.abs(np.fft.rfft(rec.channels["Bx"] * taper)) ** 2
ax_r.semilogy(freq / MHz, power / power.max(), lw=0.8, color="#3a7bd5")
for edge in aod.band:
    ax_r.axvline(edge / MHz, color="#c1121f", lw=1.0, ls="--")
ax_r.set(xlim=(85, 115), ylim=(1e-13, 3), xlabel="RF frequency [MHz]",
         ylabel="power (norm.)", title="the record: Bx, Hann-windowed (band edges dashed)")

ax_a.semilogy(nu * mm, orders / orders.max(), lw=0.8, color="#3a7bd5")
window = band_window(nu, centre, half, 0.25)
ax_a.fill_between(nu * mm, 1e-16, np.where(window > 0, 3, 1e-16), color="#8ac926", alpha=0.25,
                  label="band window (+1 order)")
for p in (-2, -1, 0, 1, 2):
    ax_a.axvline(p * centre * mm, color="#5a6472", lw=0.7, ls=":")
    ax_a.text(p * centre * mm, 2.0, f"{p:+d}", ha="center", fontsize=8, color="#5a6472")
ax_a.set(xlim=(-2.6 * abs(centre) * mm, 2.6 * abs(centre) * mm), ylim=(1e-13, 6),
         xlabel=r"aperture spatial frequency [mm$^{-1}$]",
         title=r"the aperture: the $e^{iCV}$ order comb at $du=\Lambda/8$")
ax_a.legend(fontsize=8, loc="lower right")
fig.tight_layout()

## 4. The crystal compresses, and by how much

Keep the $+1$ order of $e^{iCA\cos\psi}$ and Jacobi–Anger gives $J_1(CA)$ where the linear
(weak-drive) model gives $CA/2$. So a single driven channel's **intensity** comes out

$$\frac{I_\text{bragg}}{I_\text{weak}} = \Big(\frac{2J_1(C)}{C}\Big)^{2} = 1 - \frac{C^2}{4} + \mathcal{O}(C^4).$$

Measured below straight from rendered samples, on one gated `Ay` tone, with no model in the
loop but the Bessel function on the right-hand side. At the product default $C = 0.30$ it is a
**2.2 %** intensity loss per driven channel — and an array's tones *stack*, which is why the
flagship's effective modulation index is 1.4 rad rather than 0.3 and its per-trap spread is a
fifth rather than a percent.

In [ ]:
from dataclasses import replace

from aodl.check import axis_pupil, zoom_field
from aodl.poly import PiecewisePoly
from aodl.waveform.tones import ChannelWaveform, SmoothOnOff, ToneTrack, WaveformSet

SPAN, DETUNING = 60.0 * us, 3.0 * MHz


def peak_ratio(strength):
    "Bragg / weak peak intensity of one gated Ay tone, rebuilt from its own samples."
    hardware = replace(P, channels={n: replace(a, drive_strength=strength)
                                    for n, a in P.channels.items()})
    tone = ToneTrack(freq=PiecewisePoly.constant(DETUNING, 0.0, SPAN),
                     env=SmoothOnOff(t_on=0.0, t_off=SPAN, ramp=8 * us))
    wfs = WaveformSet({"Ay": ChannelWaveform((tone,))}, hardware)
    buf, sc = render_samples(wfs, DEFAULT_SAMPLE_RATE, (0.0, SPAN), dtype=np.float64,
                             return_scale=True)
    beat = demodulate(from_arrays(buf, DEFAULT_SAMPLE_RATE, hardware, normalization=sc))
    g = ApertureGrid.design(hardware, "bragg_band")
    ys = np.linspace(-hardware.deflection_scale * DETUNING - 2 * P.optics.waist0,
                     -hardware.deflection_scale * DETUNING + 2 * P.optics.waist0, 81)
    intensity = {m: np.abs(zoom_field(axis_pupil(beat, "y", 2 * tau, g, mode=m), g,
                                      hardware.optics, ys, 0.0)).max() ** 2
                 for m in ("bragg_band", "weak")}
    return intensity["bragg_band"] / intensity["weak"]


strengths = np.array([0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.2])
measured = np.array([peak_ratio(float(c)) for c in strengths])
predicted = (2.0 * j1(strengths) / strengths) ** 2
print(f"worst departure from (2 J1(C)/C)^2 : {np.abs(measured / predicted - 1).max():.2e}")
print(f"loss at the product default C = 0.30: {100 * (1 - measured[4]):.2f} %")

In [ ]:
fig, (ax, ax_r) = plt.subplots(1, 2, figsize=(11.0, 3.6))
fine = np.linspace(0.01, 1.3, 200)
ax.plot(fine, (2 * j1(fine) / fine) ** 2, color="k", lw=2.5, alpha=0.25,
        label=r"$(2J_1(C)/C)^2$")
ax.plot(strengths, measured, "o", color="#3a7bd5", label="measured from samples")
ax.axvline(0.3, color="#c1121f", lw=0.8, ls="--")
ax.set(xlabel="drive strength $C$ [rad]", ylabel=r"$I_{\rm bragg}/I_{\rm weak}$",
       title="compression of the fundamental (dashed: the product default)")
ax.legend(fontsize=8)

ax_r.semilogy(strengths, np.abs(measured / predicted - 1), "o-", color="#3a7bd5", lw=1.0)
ax_r.set(xlabel="drive strength $C$ [rad]", ylabel="relative departure",
         title="measurement vs Bessel, over two decades of $C$")
fig.tight_layout()

## 5. Measured vs requested, with the retardation put back

The atom plane lags the drive by $\tau/2 = 5.77$ µs (`docs/conventions.md` §7), so the honest
comparison at observation time $t$ is against the trajectory at $t - \tau/2$ — which is exactly
what `Expectation.eval_time` does, and what the `dx`/`dy`/`dz` columns of the report's table
already carry. Left: where the array centre went. Right: the residual of every one of the 700
(frame, trap) rows against its gate.

In [ ]:
table = report.table
x_law, y_law, z_law = story.compile()
lag = np.clip(report.times - 0.5 * tau, 0.0, story.duration)
centre_x = np.array([table["x"][table["time"] == t].mean() for t in report.times])
centre_z = np.array([table["z_lab"][table["time"] == t].mean() for t in report.times])

fig, (ax_t, ax_e) = plt.subplots(1, 2, figsize=(11.6, 3.9))
dense = np.linspace(0.0, story.duration, 400)
ax_t.plot(dense / us + 0.5 * tau / us, np.asarray(x_law(dense)) / um, color="k", lw=3,
          alpha=0.2, label=r"requested $X(t-\tau/2)$")
ax_t.plot(dense / us + 0.5 * tau / us, np.asarray(z_law(dense)) / um, color="k", lw=3,
          alpha=0.2, ls="--", label=r"requested $Z(t-\tau/2)$")
ax_t.plot(report.times / us, centre_x / um, "o", color="#3a7bd5", label="measured $X$")
ax_t.plot(report.times / us, centre_z / um, "s", color="#f4a261", label="measured $Z$")
ax_t.set(xlabel="frame time [µs]", ylabel="array centre [µm]",
         title="the array went where it was asked to")
ax_t.legend(fontsize=8, loc="upper left")

w0, zR = P.optics.waist0, P.optics.rayleigh
gated = table["gated"] > 0.5
for key, scale, color, label in (("dx", w0, "#3a7bd5", r"$|dx|/w_0$"),
                                 ("dz", zR, "#f4a261", r"$|dz|/z_R$"),
                                 ("delta_f", zR, "#c1121f", r"$|\Delta F|/z_R$")):
    ax_e.semilogy(table["time"][gated] / us,
                  np.maximum(np.abs(table[key][gated]) / scale, 1e-8), ".", ms=4,
                  color=color, label=label)
ax_e.axhline(report.tolerances.lateral, color="#5a6472", lw=1.0, ls="--")
ax_e.text(report.times[0] / us, report.tolerances.lateral * 1.2, "tolerance band", fontsize=8,
          color="#5a6472")
ax_e.set(xlabel="frame time [µs]", ylabel="residual (relative)", ylim=(1e-6, 1),
         title="every gated (frame, trap) row against its gate")
ax_e.legend(fontsize=8, loc="lower right", ncols=3)
fig.tight_layout()

## 6. The blob audit during a hand-over

A fading ladder always has a rung on the way in or out, so an array is **wider than the one you
asked for**: $M+1$ columns at every instant for even $M$ (§6.7 of the guide), plus the Eq. S31
shadow tweezers at $\pm\,\lambda F\Delta f/v$ mid-fade. All of that lands on the *extended
lattice*, and the audit whitelists it — that is real light, at real trap depth, and the report
says so rather than pretending it is not there.

What is **not** whitelisted is light anywhere else. Off-lattice blobs are gated at 1 % of the
median trap peak, and the flagship has none at any frame.

In [ ]:
frame_index = 3
at = report.times[frame_index]
here = [b for b in report.blobs if b.time == at]
xs, ys = plan.expectation().lattice(float(at), extend=1)
traps = plan.expectation().traps(float(at))
print(f"frame {at / us:.2f} us: {len(here)} non-trap blobs, "
      f"{sum(not b.on_lattice for b in here)} of them off-lattice; "
      f"brightest on-lattice {max((b.rel_intensity for b in here), default=0):.3f}")

fig, ax = plt.subplots(figsize=(6.4, 5.6))
for x in xs:
    ax.axvline(x / um, color="#5a6472", lw=0.4, alpha=0.5)
for y in ys:
    ax.axhline(y / um, color="#5a6472", lw=0.4, alpha=0.5)
ax.plot(traps.x / um, traps.y / um, "s", ms=4, mfc="none", color="#3a7bd5",
        label="requested traps")
if here:
    ax.scatter([b.x / um for b in here], [b.y / um for b in here],
               s=[12 + 90 * b.rel_intensity for b in here],
               c=["#8ac926" if b.on_lattice else "#c1121f" for b in here],
               alpha=0.75, label="extra light (green = on lattice)")
ax.set(xlabel="X [µm]", ylabel="Y [µm]", title=f"blob audit at t = {at / us:.1f} µs "
       "(grey: the extended lattice)")
ax.legend(fontsize=8, loc="upper left")
ax.set_aspect("equal")
fig.tight_layout()

## 7. Breaking it on purpose

A checker that only ever says PASS is worth nothing. Eq. S19 splits the lateral term
antisymmetrically across a counter-propagating pair and puts the *same* $f_Z$ on all four
channels, so $X = \frac{\lambda F}{v}(f_{Bx}-f_{Ax})$ comes out right while the chirps cancel
in $\Delta F$. Negate one member's frequency law and both statements break at once — and
nothing in the synthesizer or the simulator would notice, because they would both be negating
it.

The drive below is a small 3×3 Eq. S19 story (the flagship's 93 tones are more than this
demonstration needs).

In [ ]:
from dataclasses import replace as dc_replace

from aodl.api import MotionPlan

mini = TrajectorySpec(
    array=ArraySpec(3, 3, delta_f_x=1.0 * MHz, delta_f_y=1.3 * MHz),
    moves=(Lift(4 * um, 40 * us), Translate(10 * um, 6 * um, 60 * us), Lift(-4 * um, 40 * us)),
)
good = plan_motion(mini, P)
frames = np.array([20.0 * us, 70.0 * us, 120.0 * us]) + 0.5 * tau
assert good.check(times=frames, k_subtimes=24).passed, "the honest drive passes"

flipped = ChannelWaveform(tuple(
    ToneTrack(freq=t.freq.scale(-1.0), env=t.env, phase0=t.phase0)
    for t in good.wfs.channels["Ax"].tones
))
broken = MotionPlan(spec=good.spec, params=good.params, options=good.options,
                    wfs=dc_replace(good.wfs, channels={**good.wfs.channels, "Ax": flipped}),
                    report=good.report)
verdict = broken.check(times=frames, k_subtimes=24)
print(verdict.summary())
assert not verdict.passed

`lateral` and `astigmatism`, named, with the offending trap and frame, in microns and in
waists. The two other corruptions the test suite pins behave the same way: drop one `Bx` ladder
rung and a whole column reports **missing trap**; put 5 % on a single `Bx` tone and
**uniformity** fires, and nothing else does.

### What a PASS certifies, and what it does not

It certifies position, focus, astigmatism, spot size, the intensity *pattern*, that every
requested trap is lit, and that nothing is lit off-lattice — all of it derived from the sample
buffer through a path that shares no code with the simulator.

It does **not** certify the absolute intensity: `render_samples` divides all four channels by
one global peak, and a common gain only rescales the image, so a drive rendered at half
amplitude checks out identically. That blind spot is real, it is stated in `report.notes`, and
it is pinned by a test (`tests/test_check_verdict.py`). The other exclusions the report will
tell you about when they apply: transient frames, hand-over frames, and the edge lines of a
fading array. `docs/guide.md` §5.5 is the reference.